In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# # initialie openai client
# client = OpenAI()

# msg = {
#     "role": "user",
#     "content": "how can i join the course"
# }

# res = client.responses.create(
#     model='gpt-5-nano',
#     input = msg['content'],
#     store=True
# )

In [3]:
# res.choices[0].message.content

## Retrieval and Search

In [2]:
import json

with open('combined.json', 'r') as f:
    docs = json.load(f)

In [3]:
docs[-50]

{'text': 'Use the following line instead in mounting the current volume to docker for Q4:\n`-v "/${PWD}/ollama_files:/root/.ollama"`',
 'section': 'Module 2: Open-Source LLMs',
 'question': 'Docker: Error: Docker mounted volume adds ;C to end of windows path',
 'course': 'llm-zoomcamp'}

In [4]:
# create index
import minsearch

index = minsearch.Index(
    text_fields=['text', 'section', 'question'],
    keyword_fields=["course"]
)

index.fit(docs)

In [5]:
# query knowledge base
def get_context(query):
    boost = {'question': 3.0}

    results = index.search(
        query=query,
        boost_dict=boost,
        num_results=5
    )

    return results

query = "can i use open source LLMs"
results = get_context(query)
results

[{'text': 'Yes. See module 2 and the open-ai-alternatives.md in module 1 folder.',
  'section': 'Module 1: Introduction',
  'question': 'OpenSource: Can I use open-source alternatives to OpenAI API?',
  'course': 'llm-zoomcamp'},
 {'text': 'Prior to using Ollama models in llm-zoomcamp tasks, you need to have ollama installed on your pc and the relevant LLM model downloaded with ollama from https://www.ollama.com\nTo download ollama for Ubuntu:\n``` curl -fsSL https://ollama.com/install.sh | sh ```\nTo download ollama for Mac and Windows, follow the guide on this link:\nhttps://ollama.com/download/\nOllama a number of open-source LLMs like:\nLlama3\nPhi3\nMistral and Mixtral\nGemma\nQwen\nYou can explore more models on https://ollama.com/library/\nTo download a model in Ollama, simply open command prompt and type:\n``` ollama run model_name ```\ne.g.\n``` ollama run phi3 ```\nIt will automatically download the model and you can use it same way as above for later time.\nTo use Ollama mod

### Get answers with Gemini

In [6]:
from google import genai
from google.genai.types import GenerateContentConfig, ThinkingConfig

gemini = genai.Client()

In [7]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
If the CONTEXT does not contain the answer, respond with "Not Provided".

QUESTION: {question}

CONTEXT: 
{context}
    """

    # create context string 
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    # inject into prompt
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

prompt = build_prompt(query, results)
print(prompt)

You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
If the CONTEXT does not contain the answer, respond with "Not Provided".

QUESTION: can i use open source LLMs

CONTEXT: 
section: Module 1: Introduction
question: OpenSource: Can I use open-source alternatives to OpenAI API?
answer: Yes. See module 2 and the open-ai-alternatives.md in module 1 folder.

section: Module 1: Introduction
question: OpenSource: How can I use Ollama open-source models locally on my pc without using any API?
answer: Prior to using Ollama models in llm-zoomcamp tasks, you need to have ollama installed on your pc and the relevant LLM model downloaded with ollama from https://www.ollama.com
To download ollama for Ubuntu:
``` curl -fsSL https://ollama.com/install.sh | sh ```
To download ollama for Mac and Windows, follow the guide on this link:
https://ollama.com/download/
Ollama a number of open-sour

In [8]:
def llm(prompt):
    # disable thinking for fast speed
    res = gemini.models.generate_content(
        model="gemini-2.5-flash", 
        contents=prompt
    )

    return res.text

In [9]:
# main func
def rag(query):
    results = get_context(query)
    prompt = build_prompt(query, results)
    response = llm(prompt)
    return response

In [18]:
QUERY = "can i still enroll if the course has already started?"

response = rag(QUERY)
print(response)

Yes, you can. You can still take part in the course, and you are eligible to submit the homeworks, even if you don't register. However, you won’t be able to submit some of the homeworks.

If you want to receive a certificate, you need to submit your project while submissions are still being accepted. This means submitting 2 out of 3 course projects and reviewing 3 peers’ projects by the deadline. Be aware that there will be deadlines for turning in the final projects.


### Using Elastic Search

In [11]:
# cd elastic-start-local
# start: ./start.sh
# stop ./stop.sh

from elasticsearch import Elasticsearch
import os

es = Elasticsearch(
    "http://localhost:9200", 
    api_key=os.getenv("ELASTIC_API_KEY")
)

In [12]:
print(es.info())

{'name': 'ff1d3d10361d', 'cluster_name': 'docker-cluster', 'cluster_uuid': '-qyeBCL6QROc9fiBJTnD3A', 'version': {'number': '9.2.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'ed771e6976fac1a085affabd45433234a4babeaf', 'build_date': '2025-11-27T08:06:51.614397514Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [18]:
# settings (fields + keyword)
index_config = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

In [20]:
# create index
es.indices.create(
    index="llm-zoomcamp-v2",
    body=index_config
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'llm-zoomcamp-v2'})

In [21]:
# index doc
from tqdm.auto import tqdm
for doc in tqdm(docs):
    es.index(
        index="llm-zoomcamp-v2",
        document=doc
    )

100%|██████████| 1034/1034 [00:03<00:00, 306.14it/s]


In [ ]:
# perform search
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields" #most_fields
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    res = es.search(index="llm-zoomcamp-v2", body=search_query)

    print(res)
    
    result_docs = []
    
    for hit in res['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [23]:
# same rag function with elastic search
def rag(query):
    results = elastic_search(query)
    prompt = build_prompt(query, results)
    response = llm(prompt)
    return response

In [24]:
QUERY = "can i still enroll if the course has already started?"

response = rag(QUERY)
print(response)

{'took': 64, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 405, 'relation': 'eq'}, 'max_score': 47.158062, 'hits': [{'_index': 'llm-zoomcamp-v2', '_id': '8GvS_poBrh9SFnJuQhV-', '_score': 47.158062, '_source': {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.", 'section': 'General course-related questions', 'question': 'Course - Can I still join the course after the start date?', 'course': 'data-engineering-zoomcamp'}}, {'_index': 'llm-zoomcamp-v2', '_id': '9mvS_poBrh9SFnJuQhW4', '_score': 42.920345, '_source': {'text': 'Yes, the slack channel remains open and you can ask questions there. But always sDocker containers exit code w search the channel first and second, check the FAQ (this document), most likely all your questions are already answered h